# Module 10 — Panel Data Methods
*Statistics for Econometrics — An intermediate course for researchers*

This module covers panel data regression techniques: fixed effects, random effects, two-way fixed effects, the Hausman test for FE vs RE, and practical applications to causal inference with panel data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import ols

try:
    from linearmodels.panel import PanelOLS, RandomEffects
except ImportError:
    !pip install linearmodels
    from linearmodels.panel import PanelOLS, RandomEffects

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

## 10.1 — Creating Panel Data

**Panel data** (or longitudinal data) contains observations of the same units (firms, individuals, countries) over multiple time periods. This structure allows us to control for time-invariant unobserved heterogeneity.

In [ ]:
# Generate panel dataset: 50 firms, 10 years
n_firms = 50
n_years = 10
n_obs = n_firms * n_years

np.random.seed(42)

# Create firm and year identifiers
firm_id = np.repeat(np.arange(1, n_firms + 1), n_years)
year = np.tile(np.arange(2014, 2024), n_firms)

# Firm quality (unobserved, fixed per firm, correlates with X)
# αᵢ ~ N(0, 1) for each firm, constant over time
alpha_i = np.repeat(np.random.normal(0, 1, n_firms), n_years)

# R&D spending: X = 5 + 0.3*year + 0.6*αᵢ + ε
# R&D is positively correlated with firm quality (endogeneity issue)
rd_spending = 5 + 0.3 * (year - 2014) + 0.6 * alpha_i + np.random.normal(0, 0.5, n_obs)

# Productivity: Y = 10 + 0.8*X + αᵢ + ε
# True causal effect of R&D on productivity is 0.8
productivity = 10 + 0.8 * rd_spending + alpha_i + np.random.normal(0, 1, n_obs)

# Create DataFrame with MultiIndex
data = pd.DataFrame({
    'firm_id': firm_id,
    'year': year,
    'productivity': productivity,
    'rd_spending': rd_spending
})

# Set MultiIndex on (firm_id, year)
data = data.set_index(['firm_id', 'year'])

print("="*70)
print("PANEL DATASET: 50 FIRMS, 10 YEARS (N=500)")
print("="*70)
print(f"\nShape: {data.shape}")
print(f"\nFirst 15 observations:")
print(data.head(15))
print(f"\nTrue data generating process:")
print(f"  Y_it = 10 + 0.8*X_it + αᵢ + ε_it")
print(f"  where αᵢ ~ N(0,1) is firm quality (unobserved, constant over time)")
print(f"  and X_it is correlated with αᵢ (correlation ≈ 0.6)")

## 10.2 — Pooled OLS (Biased Baseline)

**Pooled OLS** ignores the panel structure and treats all observations as independent. When firm quality αᵢ is omitted and correlated with X, pooled OLS suffers from omitted variable bias.

In [ ]:
# Flatten data for OLS (remove MultiIndex)
data_flat = data.reset_index()

# Pooled OLS: Y ~ X (ignoring panel structure)
model_pooled = ols('productivity ~ rd_spending', data=data_flat).fit()

print("="*70)
print("POOLED OLS REGRESSION (BIASED BASELINE)")
print("="*70)
print(model_pooled.summary())
print(f"\nTrue causal effect: β = 0.8")
print(f"Pooled OLS estimate: β̂ = {model_pooled.params['rd_spending']:.4f}")
print(f"Bias = {model_pooled.params['rd_spending'] - 0.8:.4f}")
print(f"\n✗ BIASED: Pooled OLS overestimates the effect because it omits αᵢ,")
print(f"  which is positively correlated with both X and Y.")

## 10.3 — Fixed Effects Estimator

**Fixed effects (FE)** regression controls for time-invariant unobserved heterogeneity (αᵢ) by including a dummy variable for each entity. This eliminates the omitted variable bias from firm quality.

In [ ]:
# Fixed effects regression using linearmodels.panel.PanelOLS
model_fe = PanelOLS(data['productivity'], data[['rd_spending']], 
                    entity_effects=True).fit(cov_type='robust')

print("="*70)
print("FIXED EFFECTS (FE) ESTIMATOR")
print("="*70)
print(model_fe.summary)
print(f"\nTrue causal effect: β = 0.8")
print(f"FE estimate: β̂ = {model_fe.params[0]:.4f}")
print(f"\n✓ FE controls for time-invariant unobserved firm quality αᵢ")
print(f"  and produces estimate closer to the true causal effect.")

In [ ]:
# Manual demeaning: demonstrate FE equivalence
# FE is equivalent to regressing demeaned Y on demeaned X

# Compute entity means
entity_mean_y = data.groupby(level='firm_id')['productivity'].transform('mean')
entity_mean_x = data.groupby(level='firm_id')['rd_spending'].transform('mean')

# Demean within each entity
y_demeaned = data['productivity'] - entity_mean_y
x_demeaned = data['rd_spending'] - entity_mean_x

# OLS on demeaned data
data_demeaned = pd.DataFrame({
    'y_demeaned': y_demeaned,
    'x_demeaned': x_demeaned
})

model_manual_fe = ols('y_demeaned ~ x_demeaned - 1', data=data_demeaned).fit()

print("="*70)
print("FIXED EFFECTS BY MANUAL DEMEANING")
print("="*70)
print(f"\nManual demeaning approach:")
print(f"  1. Subtract firm means from Y and X")
print(f"  2. Regress demeaned Y on demeaned X (no intercept)")
print(f"\nFE estimate (manual demeaning): β̂ = {model_manual_fe.params['x_demeaned']:.4f}")
print(f"FE estimate (PanelOLS):          β̂ = {model_fe.params[0]:.4f}")
print(f"\nDifference: {abs(model_manual_fe.params['x_demeaned'] - model_fe.params[0]):.6f}")
print(f"✓ Manual demeaning confirms FE equivalence principle.")

## 10.4 — Random Effects Estimator

**Random effects (RE)** assumes firm quality αᵢ is uncorrelated with X. Under this assumption, RE is more efficient (lower standard errors) than FE. However, if the assumption is violated (as in this data), RE is biased.

In [ ]:
# Random effects regression
model_re = RandomEffects(data['productivity'], data[['rd_spending']]).fit(cov_type='robust')

print("="*70)
print("RANDOM EFFECTS (RE) ESTIMATOR")
print("="*70)
print(model_re.summary)
print(f"\nTrue causal effect: β = 0.8")
print(f"RE estimate: β̂ = {model_re.params[0]:.4f}")
print(f"\nComparison:")
print(f"  Pooled OLS: β̂ = {model_pooled.params['rd_spending']:.4f} (biased)")
print(f"  FE:         β̂ = {model_fe.params[0]:.4f}")
print(f"  RE:         β̂ = {model_re.params[0]:.4f}")
print(f"\nRE estimate is between pooled OLS and FE because firm quality")
print(f"is actually correlated with X (violates RE assumption).")

## 10.5 — Hausman Test: FE vs RE

The **Hausman test** compares FE and RE estimators to test whether the RE assumption (no correlation between αᵢ and X) is valid.

H₀: RE is appropriate (αᵢ uncorrelated with X)  
H₁: FE is appropriate (αᵢ correlated with X)

The test statistic is: H = (β̂_FE - β̂_RE)² / (Var(β̂_FE) - Var(β̂_RE)) ~ χ²(1)

In [ ]:
# Hausman test: FE vs RE
# Manual computation

beta_fe = model_fe.params[0]
beta_re = model_re.params[0]
se_fe = model_fe.std_errors[0]
se_re = model_re.std_errors[0]

beta_diff = beta_fe - beta_re
var_diff = se_fe**2 - se_re**2

# Hausman test statistic
h_stat = (beta_diff**2) / var_diff

# P-value from chi-squared distribution with 1 degree of freedom
p_value = 1 - stats.chi2.cdf(h_stat, df=1)

print("="*70)
print("HAUSMAN TEST: FE vs RE")
print("="*70)
print(f"\nFE estimate: β̂_FE = {beta_fe:.4f} (SE = {se_fe:.4f})")
print(f"RE estimate: β̂_RE = {beta_re:.4f} (SE = {se_re:.4f})")
print(f"\nDifference: β̂_FE - β̂_RE = {beta_diff:.4f}")
print(f"Var(β̂_FE) - Var(β̂_RE) = {var_diff:.6f}")
print(f"\nHausman statistic: H = {h_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nCritical value (χ²₀.₀₅, df=1): {stats.chi2.ppf(0.95, df=1):.4f}")

if p_value < 0.05:
    print(f"\n✓ REJECT H₀ at α=0.05")
    print(f"  Conclusion: Firm quality is correlated with R&D spending.")
    print(f"  Use FE estimates (not RE).")
else:
    print(f"\n✗ FAIL TO REJECT H₀ at α=0.05")
    print(f"  Conclusion: RE is appropriate.")

## 10.6 — Two-Way Fixed Effects

**Two-way fixed effects (2FE)** controls for both entity-specific effects (αᵢ) and time-specific effects (λₜ). This accounts for time-invariant unobserved heterogeneity across firms AND across time periods.

In [ ]:
# Add time-specific shocks to the data
# Y_it = 10 + 0.8*X_it + αᵢ + λₜ + ε_it
# where λₜ are time-specific effects

data_with_time_shock = data.copy()

# Add time effects: λₜ = 0.1*(year - 2014) - small positive trend
time_effect = (data_with_time_shock.index.get_level_values('year') - 2014) * 0.1
data_with_time_shock['productivity'] = data_with_time_shock['productivity'] + time_effect

# One-way FE (only entity effects)
model_oneway_fe = PanelOLS(data_with_time_shock['productivity'], 
                           data_with_time_shock[['rd_spending']], 
                           entity_effects=True).fit(cov_type='robust')

# Two-way FE (entity + time effects)
model_twoway_fe = PanelOLS(data_with_time_shock['productivity'], 
                           data_with_time_shock[['rd_spending']], 
                           entity_effects=True, time_effects=True).fit(cov_type='robust')

print("="*70)
print("TWO-WAY FIXED EFFECTS")
print("="*70)
print(f"\nOne-way FE (entity effects only):")
print(f"  β̂ = {model_oneway_fe.params[0]:.4f}")
print(f"  SE = {model_oneway_fe.std_errors[0]:.4f}")

print(f"\nTwo-way FE (entity + time effects):")
print(f"  β̂ = {model_twoway_fe.params[0]:.4f}")
print(f"  SE = {model_twoway_fe.std_errors[0]:.4f}")

print(f"\nTrue causal effect: β = 0.8")
print(f"\nTwo-way FE controls for both firm-specific and time-specific")
print(f"unobserved heterogeneity, providing more robust estimates.")

## 10.7 — Clustered Standard Errors

In panel data, observations are correlated within clusters (same firm over time). **Clustered standard errors** account for this intra-cluster correlation, typically yielding larger standard errors than non-clustered standard errors.

In [ ]:
# Compare standard errors: non-clustered vs clustered at entity level

# Non-clustered standard errors
model_fe_nocluster = PanelOLS(data['productivity'], data[['rd_spending']], 
                              entity_effects=True).fit(cov_type='homoskedastic')

# Clustered standard errors (default is cluster by entity)
model_fe_cluster = PanelOLS(data['productivity'], data[['rd_spending']], 
                            entity_effects=True).fit(cov_type='robust')

print("="*70)
print("CLUSTERED STANDARD ERRORS")
print("="*70)
print(f"\nNon-clustered standard errors:")
print(f"  β̂ = {model_fe_nocluster.params[0]:.4f}")
print(f"  SE = {model_fe_nocluster.std_errors[0]:.4f}")

print(f"\nClustered standard errors (by firm):")
print(f"  β̂ = {model_fe_cluster.params[0]:.4f}")
print(f"  SE = {model_fe_cluster.std_errors[0]:.4f}")

se_ratio = model_fe_cluster.std_errors[0] / model_fe_nocluster.std_errors[0]
print(f"\nRatio (clustered / non-clustered): {se_ratio:.3f}")
print(f"\n✓ Clustered SEs are {se_ratio:.1f}x larger, reflecting correlation")
print(f"  within firms over time.")

## 10.8 — Summary Comparison

This section compares all panel data estimation methods side-by-side.

In [ ]:
# Create summary comparison table

summary_table = pd.DataFrame({
    'Estimator': [
        'Pooled OLS',
        'Fixed Effects (FE)',
        'Random Effects (RE)',
        'Two-Way FE'
    ],
    'Coefficient': [
        f"{model_pooled.params['rd_spending']:.4f}",
        f"{model_fe.params[0]:.4f}",
        f"{model_re.params[0]:.4f}",
        f"{model_twoway_fe.params[0]:.4f}"
    ],
    'Std. Error': [
        f"{model_pooled.bse['rd_spending']:.4f}",
        f"{model_fe.std_errors[0]:.4f}",
        f"{model_re.std_errors[0]:.4f}",
        f"{model_twoway_fe.std_errors[0]:.4f}"
    ],
    'Controls for αᵢ?': ['No', 'Yes', 'Assumes uncorr', 'Yes'],
    'Controls for λₜ?': ['No', 'No', 'No', 'Yes']
})

print("="*100)
print("SUMMARY: COMPARISON OF PANEL DATA ESTIMATORS")
print("="*100)
print(f"\nTrue causal effect: β = 0.8")
print(f"\n{summary_table.to_string(index=False)}")

print(f"\n\nKEY INSIGHTS:")
print(f"1. Pooled OLS is BIASED upward because firm quality αᵢ is omitted")
print(f"   and correlated with X.")
print(f"\n2. Fixed Effects removes the bias by controlling for αᵢ")
print(f"   (time-invariant unobserved firm quality).")
print(f"\n3. Random Effects assumes αᵢ ⊥ X, which is violated here,")
print(f"   so RE is biased in this example.")
print(f"\n4. Hausman test rejects RE (p < 0.05), confirming that")
print(f"   FE is the appropriate choice.")
print(f"\n5. Two-Way FE additionally controls for time-specific shocks λₜ,")
print(f"   removing confounding from temporal trends.")